## imports:

In [18]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import pandas as pd
import numpy as np

pd.set_option('display.max_columns',None)
print("IMPORTS READY")

IMPORTS READY


## 1. load and merge

In [19]:
import pandas as pd

results_df = pd.read_csv('../data/raw/results.csv')
wc2026_df  = pd.read_csv('../data/wc2026/wc2026_new_results.csv')

results_df['date'] = pd.to_datetime(results_df['date'])
wc2026_df['date']  = pd.to_datetime(wc2026_df['date'])

print(f"results.csv        : {len(results_df)} rows, max date {results_df['date'].max().date()}")
print(f"wc2026_new_results : {len(wc2026_df)} rows, max date {wc2026_df['date'].max().date()}")

# Align columns explicitly, tag source so we can resolve overlaps deliberately
wc2026_aligned = wc2026_df[results_df.columns.tolist()].copy()
results_df = results_df.copy()
results_df['source'] = 'historical'
wc2026_aligned['source'] = 'wc2026_manual'

combined_df = pd.concat([results_df, wc2026_aligned], ignore_index=True)
print(f"\nCombined (before dedup): {len(combined_df)} rows")

results.csv        : 49477 rows, max date 2026-06-27
wc2026_new_results : 96 rows, max date 2026-07-07

Combined (before dedup): 49573 rows


## 2. Resolve duplicates:

In [20]:
dupe_mask = combined_df.duplicated(subset=['date','home_team','away_team'], keep=False)
print(f"Total duplicate rows found: {dupe_mask.sum()}")

overlap_keys = set(
    combined_df[dupe_mask & (combined_df['source']=='wc2026_manual')][['date','home_team','away_team']]
    .apply(tuple, axis=1)
)

def is_stale_historical(row):
    return row['source'] == 'historical' and (row['date'], row['home_team'], row['away_team']) in overlap_keys

drop_mask = combined_df.apply(is_stale_historical, axis=1)
print(f"Dropping stale historical rows superseded by wc2026 backfill: {drop_mask.sum()}")
combined_df = combined_df[~drop_mask].copy()

# Exact duplicate rows (identical everything)
before = len(combined_df)
combined_df = combined_df.drop_duplicates(subset=['date','home_team','away_team','home_score','away_score'])
print(f"Dropped exact duplicate rows: {before - len(combined_df)}")

# Remaining conflicting duplicates — can't verify which is correct, drop both
still_dup = combined_df.duplicated(subset=['date','home_team','away_team'], keep=False)
print(f"Remaining unresolved conflicting duplicates: {still_dup.sum()}")
if still_dup.sum() > 0:
    print(combined_df[still_dup][['date','home_team','away_team','home_score','away_score','source']])

combined_df = combined_df[~still_dup].copy()
combined_df = combined_df.drop(columns=['source'])

print(f"\nFinal combined_df shape: {combined_df.shape}")
print(f"Date range: {combined_df['date'].min().date()} to {combined_df['date'].max().date()}")

Total duplicate rows found: 132
Dropping stale historical rows superseded by wc2026 backfill: 64
Dropped exact duplicate rows: 1
Remaining unresolved conflicting duplicates: 2
           date home_team      away_team  home_score  away_score      source
9641 1974-02-17    Tahiti  New Caledonia         2.0         1.0  historical
9642 1974-02-17    Tahiti  New Caledonia         1.0         2.0  historical

Final combined_df shape: (49506, 9)
Date range: 1872-11-30 to 2026-07-07


## 3. Apply tournament + date filter

In [21]:

old_clean = pd.read_csv('/Users/souranilchakraborty/Desktop/ROADMAP/Hands-on_folder/FIFA/data/clean/clean_matches1.csv') 
tournaments_to_keep = sorted(old_clean['tournament'].unique())
print(f"Tournament whitelist ({len(tournaments_to_keep)}): {tournaments_to_keep}")

new_clean = combined_df[
    (combined_df['date'] >= '2000-01-01') &
    (combined_df['tournament'].isin(tournaments_to_keep))
].copy()

new_clean = new_clean.dropna(subset=['home_score','away_score']).reset_index(drop=True)

print(f"\nOLD clean_matches.csv : {len(old_clean)} rows")
print(f"NEW clean_matches.csv : {len(new_clean)} rows")
print(f"New matches added     : {len(new_clean) - len(old_clean)}")
print(f"Null check             : {new_clean[['home_score','away_score']].isnull().sum().sum()}")

Tournament whitelist (19): ['AFC Asian Cup', 'AFC Asian Cup qualification', 'African Cup of Nations', 'African Cup of Nations qualification', 'Amílcar Cabral Cup', 'Arab Cup', 'CAFA Nations Cup', 'CONCACAF Nations League', 'CONCACAF Nations League qualification', 'Confederations Cup', 'Copa América', 'FIFA World Cup', 'FIFA World Cup qualification', 'Gold Cup', 'Oceania Nations Cup', 'UEFA Euro', 'UEFA Euro qualification', 'UEFA Nations League', 'WAFF Championship']

OLD clean_matches.csv : 12956 rows
NEW clean_matches.csv : 13052 rows
New matches added     : 96
Null check             : 0


## 4. Team Name consisteny check

In [22]:
historical = new_clean[new_clean['date'] < '2026-06-01']
canonical_teams = set(pd.concat([historical['home_team'], historical['away_team']]).unique())

wc2026_matches = new_clean[new_clean['date'] >= '2026-06-01']
wc2026_teams_used = set(pd.concat([wc2026_matches['home_team'], wc2026_matches['away_team']]).unique())

not_in_historical = sorted(wc2026_teams_used - canonical_teams)
print(f"Team names in 2026 matches NOT matching historical convention: {not_in_historical}")

for t in not_in_historical:
    n = ((wc2026_matches['home_team']==t) | (wc2026_matches['away_team']==t)).sum()
    print(f"  {t!r} — {n} matches")

Team names in 2026 matches NOT matching historical convention: ['Bosnia & Herzegovina', 'USA']
  'Bosnia & Herzegovina' — 4 matches
  'USA' — 5 matches


## 5. Fix names and save final:

In [23]:
name_fixes = {
    'USA': 'United States',
    'Bosnia & Herzegovina': 'Bosnia and Herzegovina'
}

for col in ['home_team', 'away_team']:
    new_clean[col] = new_clean[col].replace(name_fixes)

# Re-verify — should now be zero
historical = new_clean[new_clean['date'] < '2026-06-01']
canonical_teams = set(pd.concat([historical['home_team'], historical['away_team']]).unique())
wc2026_matches = new_clean[new_clean['date'] >= '2026-06-01']
wc2026_teams_used = set(pd.concat([wc2026_matches['home_team'], wc2026_matches['away_team']]).unique())
remaining_mismatches = sorted(wc2026_teams_used - canonical_teams)

print(f"Remaining mismatches after fix: {len(remaining_mismatches)}")
print(remaining_mismatches)

Remaining mismatches after fix: 0
[]


## 6. FUll 48 - team 

In [24]:
wc2026_teams = [
    'Mexico', 'South Africa', 'Czech Republic', 'South Korea',
    'Canada', 'Switzerland', 'Bosnia and Herzegovina', 'Qatar',
    'Brazil', 'Morocco', 'Haiti', 'Scotland',
    'United States', 'Australia', 'Greece', 'Cameroon',
    'Germany', 'Japan', 'Ecuador', 'Iraq',
    'Netherlands', 'New Zealand', 'Iran', 'Senegal',
    'Belgium', 'Egypt', 'Saudi Arabia', 'Curaçao',
    'Spain', 'Uruguay', 'Cape Verde', 'Ivory Coast',
    'France', 'Norway', 'Algeria', 'Jordan',
    'Argentina', 'Austria', 'Venezuela', 'DR Congo',
    'Portugal', 'Colombia', 'Uzbekistan', 'Slovenia',
    'England', 'Croatia', 'Ghana', 'Panama',
    'Paraguay', 'Turkey', 'Tunisia'
]

all_teams_in_new = set(pd.concat([new_clean['home_team'], new_clean['away_team']]).unique())
missing = [t for t in wc2026_teams if t not in all_teams_in_new]

print(f"WC2026 teams missing entirely: {len(missing)}")
print(missing)

# Specifically confirm tomorrow's 8 quarterfinalists have clean recent data
qf_teams = ['France', 'Morocco', 'Spain', 'Belgium', 'Norway', 'England', 'Argentina', 'Switzerland']
for team in qf_teams:
    recent = new_clean[
        ((new_clean['home_team']==team) | (new_clean['away_team']==team)) &
        (new_clean['date'] >= '2026-06-01')
    ]
    print(f"{team:<12} — {len(recent)} matches in 2026, most recent: {recent['date'].max().date() if len(recent) else 'NONE'}")

WC2026 teams missing entirely: 0
[]
France       — 5 matches in 2026, most recent: 2026-07-04
Morocco      — 5 matches in 2026, most recent: 2026-07-04
Spain        — 5 matches in 2026, most recent: 2026-07-06
Belgium      — 5 matches in 2026, most recent: 2026-07-06
Norway       — 5 matches in 2026, most recent: 2026-07-05
England      — 5 matches in 2026, most recent: 2026-07-05
Argentina    — 5 matches in 2026, most recent: 2026-07-07
Switzerland  — 5 matches in 2026, most recent: 2026-07-07


## 7. save :

In [25]:
new_clean.to_csv('../data/processed/clean_matches.csv', index=False)
print(f"Saved — shape: {new_clean.shape}")
print(f"Date range: {new_clean['date'].min().date()} to {new_clean['date'].max().date()}")

Saved — shape: (13052, 9)
Date range: 2000-01-22 to 2026-07-07


## 8. `get_shootout_win_rate` function:
shrunk_rate = (wins + k × prior) / (n + k)

prior = 0.5   (neutral — no historical edge)

k     = 3     (how many "phantom neutral observations" we blend in)

In [26]:
def build_shootout_index(shootouts_df):
    """
    {team_name: DataFrame[date, won]} — one row per shootout that team was in,
    sorted by date, 'won' is True/False.
    """
    shootouts_df = shootouts_df.sort_values('date').reset_index(drop=True)
    index = {}

    all_teams = pd.concat([shootouts_df['home_team'], shootouts_df['away_team']]).unique()

    for team in all_teams:
        mask = (shootouts_df['home_team'] == team) | (shootouts_df['away_team'] == team)
        team_shootouts = shootouts_df[mask].copy()
        team_shootouts['won'] = team_shootouts['winner'] == team
        index[team] = team_shootouts[['date', 'won']].reset_index(drop=True)

    return index


def get_shootout_win_rate(team, date, shootout_index, k=3, prior=0.5):

    if team not in shootout_index:
        return prior  # no shootout history at all — fully neutral

    team_shootouts = shootout_index[team]
    prior_shootouts = team_shootouts[team_shootouts['date'] < date]

    n = len(prior_shootouts)
    if n == 0:
        return prior

    wins = prior_shootouts['won'].sum()
    shrunk_rate = (wins + k * prior) / (n + k)

    return round(shrunk_rate, 4)

In [27]:
shootouts_df = pd.read_csv('../data/raw/shootouts_new.csv', parse_dates=['date'])
shootout_index = build_shootout_index(shootouts_df)

test_date = pd.Timestamp('2026-07-09')  # tomorrow, before kickoff

for team in ['Morocco', 'Switzerland', 'France', 'Spain', 'Belgium', 'Norway', 'England', 'Argentina']:
    rate = get_shootout_win_rate(team, test_date, shootout_index)
    n_shootouts = len(shootout_index[team][shootout_index[team]['date'] < test_date]) if team in shootout_index else 0
    print(f"{team:<12} — n={n_shootouts}, shrunk_win_rate={rate}")

Morocco      — n=15, shrunk_win_rate=0.4722
Switzerland  — n=9, shrunk_win_rate=0.375
France       — n=11, shrunk_win_rate=0.4643
Spain        — n=14, shrunk_win_rate=0.5
Belgium      — n=2, shrunk_win_rate=0.7
Norway       — n=0, shrunk_win_rate=0.5
England      — n=12, shrunk_win_rate=0.3667
Argentina    — n=23, shrunk_win_rate=0.6346


In [29]:
### checking features.py
import sys
sys.path.append('code')
from features import (
    build_team_match_index, build_goals_index,
    get_team_form_fast, get_attack_defense_fast,
    get_clean_sheet_rate, get_days_rest,
    get_second_half_goals_pct, get_penalty_goal_pct,
    get_h2h_fast, calculate_elo_history, get_elo_at_date,
    build_shootout_index, get_shootout_win_rate,
    calculate_lambda, build_score_matrix
)
print("✅ features.py loaded — 15 functions, self-contained")

✅ features.py loaded — 15 functions, self-contained
